In [1]:
# Setting Up LiteLLM and PromptFoo Integration

"""
This guide provides a complete implementation for testing and evaluating LLM prompts
using LiteLLM and PromptFoo. The implementation includes environment setup, 
prompt template creation, test case definition, and evaluation analysis.
"""

# Part 1: Environment Setup
# First, install required packages:
# pip install litellm promptfoo python-dotenv
# npm install -g promptfoo

import os
from dotenv import load_dotenv
from litellm import completion
import json
import pandas as pd
import subprocess
import matplotlib.pyplot as plt

# Load environment variables
load_dotenv()

# Part 2: LiteLLM Implementation
class LLMTester:
    """
    A class to manage LLM testing and evaluation using LiteLLM and PromptFoo.
    Provides methods for model interaction, test case management, and result analysis.
    """
    
    def __init__(self):
        self.models = {
            "anthropic": "claude-3-sonnet-20240229",
            "openai": "gpt-4"
        }
        
    def get_model_response(self, prompt, model="claude-3-sonnet-20240229"):
        """
        Get response from different LLM models using LiteLLM
        
        Args:
            prompt (str): The input prompt
            model (str): Model identifier (e.g., 'gpt-4', 'claude-3-sonnet-20240229')
            
        Returns:
            str: Model's response
        """
        try:
            response = completion(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.7
            )
            return response.choices[0].message.content
        except Exception as e:
            return f"Error: {str(e)}"

    def create_promptfoo_config(self):
        """
        Creates and saves PromptFoo configuration with test templates
        """
        config = {
            "providers": [
                {"id": "anthropic", "model": self.models["anthropic"]},
                {"id": "openai", "model": self.models["openai"]}
            ],
            "prompts": [
                # Customer Service Template
                """You are a customer service representative. Help resolve the following issue:
                Customer message: {{customer_message}}
                
                Respond professionally and empathetically.""",
                
                # Product Description Template
                """Create a compelling product description for:
                Product: {{product_name}}
                Features: {{features}}
                Target audience: {{audience}}"""
            ]
        }
        
        with open('promptfoo.json', 'w') as f:
            json.dump(config, f, indent=2)
        
        return config

    def create_test_cases(self):
        """
        Creates and saves test cases with expected outputs
        """
        test_cases = {
            "tests": [
                {
                    "vars": {
                        "customer_message": "I haven't received my order that was supposed to arrive yesterday."
                    },
                    "assert": {
                        "contains": ["apologize", "track", "help"],
                        "excludes": ["sorry for the confusion", "let me know if"],
                        "max_tokens": 200
                    }
                },
                {
                    "vars": {
                        "product_name": "EcoFresh Water Bottle",
                        "features": "Vacuum insulated, 24-hour temperature control, sustainable materials",
                        "audience": "Outdoor enthusiasts and eco-conscious consumers"
                    },
                    "assert": {
                        "contains": ["sustainable", "temperature", "outdoor"],
                        "tone": "professional",
                        "max_tokens": 150
                    }
                }
            ]
        }
        
        with open('test_cases.json', 'w') as f:
            json.dump(test_cases, f, indent=2)
        
        return test_cases

    def run_evaluation(self):
        """
        Runs PromptFoo evaluation and returns results as DataFrame
        """
        try:
            result = subprocess.run(
                ['promptfoo', 'eval', 
                 '--config', 'promptfoo.json', 
                 '--tests', 'test_cases.json', 
                 '--output', 'results.json'],
                capture_output=True,
                text=True
            )
            
            with open('results.json', 'r') as f:
                results = json.load(f)
                
            return pd.DataFrame(results['results'])
            
        except Exception as e:
            print(f"Error running evaluation: {str(e)}")
            return None

    def analyze_results(self, df_results):
        """
        Analyzes and visualizes evaluation results
        
        Args:
            df_results (pd.DataFrame): Evaluation results DataFrame
        """
        if df_results is None:
            return
        
        print("=== Evaluation Summary ===")
        print(f"Total tests run: {len(df_results)}")
        print(f"Pass rate: {(df_results['pass'].sum() / len(df_results)) * 100:.2f}%")
        
        # Model comparison
        model_stats = df_results.groupby('provider')['pass'].agg(['count', 'mean'])
        print("\n=== Model Performance ===")
        print(model_stats)
        
        # Visualize results
        plt.figure(figsize=(10, 6))
        model_stats['mean'].plot(kind='bar')
        plt.title('Model Success Rate Comparison')
        plt.ylabel('Success Rate')
        plt.tight_layout()
        plt.show()
        
        # Save detailed results
        df_results.to_csv('detailed_results.csv', index=False)
        print("\nDetailed results saved to 'detailed_results.csv'")

# Usage Example
def main():
    # Initialize tester
    tester = LLMTester()
    
    # Create configuration and test cases
    tester.create_promptfoo_config()
    tester.create_test_cases()
    
    # Run evaluation
    results = tester.run_evaluation()
    
    # Analyze results
    tester.analyze_results(results)

if __name__ == "__main__":
    main()

Error running evaluation: [Errno 2] No such file or directory: 'promptfoo'


In [ ]:
from llm_tester import LLMTester

tester = LLMTester()
tester.create_promptfoo_config()
tester.create_test_cases()
results = tester.run_evaluation()
tester.analyze_results(results)